# 01 · Data quality: March 2023 one-minute OHLCV

Before any liquidity statistic is computed, every exchange feed is audited:

1. **Activity** – is the market actually trading, or is the venue publishing stale placeholder quotes?
2. **Gaps** – how many minutes are missing, and how are they filled (causally, with no look-ahead)?

Gap-filling rule (see `stablecoin_risk.data.cleaning`): ≤5 missing bars → last observation carried forward;
6–30 → simple-exponential-smoothing level fitted on the prior day; >30 → left missing and logged.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import config
from stablecoin_risk.data import loaders, cleaning

raw = loaders.load_svb_panel(source="raw", drop_inactive=False)
activity = pd.DataFrame({
    k: {"bars": len(d), "zero_volume_%": 100 * (d["volume"] == 0).mean(),
        "distinct_closes": d["close"].nunique(), "active": loaders.is_active_market(d)}
    for k, d in raw.items()}).T
activity

,bars,zero_volume_%,distinct_closes,active
binance_BTCUSD,30240,0.2381,28115,True
binance_BTCUSDC,30240,42.89,15723,True
binance_BTCUSDT,30240,2.176,27440,True
binance_USDCUSD,30240,65.4,740,True
binance_USDTUSD,30240,19.95,171,True
coinbase_BTCEUR,27261,0,25557,True
coinbase_BTCUSD,29950,0,28184,True
coinbase_BTCUSDC,28511,0,26794,True
coinbase_BTCUSDT,23349,0,21595,True
coinbase_USDCEUR,8276,0,810,True


**Two dead feeds.** Crypto.com BTC/USDC and USDT/USDC report zero volume in every minute and a single frozen
price (BTC/USDC is stuck at 20,100 while BTC traded between 19,600 and 28,500). An early draft of this research
read the resulting price difference as a "42% BTC/USDC arbitrage gap" between Binance and Crypto.com; it is a data
artifact, so these markets are excluded by `is_active_market` everywhere downstream.

**One mirrored feed.** Coinbase merged its USD and USDC order books in 2022, so `coinbase_BTCUSDC` is the BTC-USD
book quoted 1:1 (its basis to BTC/USD is exactly zero). It is listed in `config.MIRRORED_MARKETS` and excluded
from per-market statistics, but kept as the "USD side" in cross-venue BTC/USDC comparisons.

In [3]:
reports = cleaning.clean_directory(config.RAW_DIR, config.PROCESSED_DIR, verbose=False)
summary = cleaning.summarize_reports(reports)
summary.assign(imputed_share=lambda d: (100 * d["imputed_share"]).round(1)).rename(columns={"imputed_share": "imputed_%"})

,original_bars,locf_bars,ets_bars,imputed_%,long_gaps,long_gap_bars
series,,,,,,
binance_BTCUSD_1m_20230301_20230321,30240,0,0,0,0,0
binance_BTCUSDC_1m_20230301_20230321,30240,0,0,0,0,0
binance_BTCUSDT_1m_20230301_20230321,30240,0,0,0,0,0
binanceus_USDCUSD_1m_20230301_20230321,30240,0,0,0,0,0
binanceus_USDTUSD_1m_20230301_20230321,30240,0,0,0,0,0
coinbase_BTCEUR_1m_20230301_20230321,27261,2604,88,9,1,287
coinbase_BTCUSD_1m_20230301_20230321,29950,13,0,0,1,277
coinbase_BTCUSDC_1m_20230301_20230321,28511,13,0,0,1,277
coinbase_BTCUSDT_1m_20230301_20230321,23349,4997,173,18.1,1,282


**Reading the table.** Binance and Crypto.com BTC books are complete. Coinbase BTC/USDT needs 18% imputation and
Coinbase's stablecoin crosses (USDC/EUR, USDT/EUR, USDT/USDC) are mostly synthetic after filling, which is why the
cross-venue comparisons in notebook 02 use **original (non-imputed, traded) bars only**
(`microstructure.original_close`), and why Coinbase is excluded from the Market Quality Index.

In [4]:
pd.read_csv(config.PROCESSED_DIR / "long_gaps_dropped.csv").groupby("series")["n_missing_bars"].agg(["count", "sum", "max"])

,count,sum,max
series,,,
coinbase_BTCEUR_1m_20230301_20230321,1,287,287
coinbase_BTCUSDC_1m_20230301_20230321,1,277,277
coinbase_BTCUSDT_1m_20230301_20230321,1,282,282
coinbase_BTCUSD_1m_20230301_20230321,1,277,277
coinbase_USDCEUR_1m_20230301_20230321,31,1628,386
coinbase_USDTEUR_1m_20230301_20230321,5,478,331
coinbase_USDTUSDC_1m_20230301_20230321,57,2489,326
